# 🔑 Solutions: Lesson 09 · Project 2 · Image Classification

> **Instructor's note.** Don't read these until you've really tried! For each problem: read the statement, look at the solution, then *close the notebook and re-implement it from memory* in the lesson notebook. The self-checks below are identical to the ones you saw there.

Each solution cell is self-contained and runs top-to-bottom.

In [1]:
import sys; sys.path.append("..")
import numpy as np
import matplotlib.pyplot as plt
from utils import dlcourse as dl
import pandas as pd, math
try:
    import torch, torch.nn as nn, torch.nn.functional as F
    from torch.utils.data import TensorDataset, DataLoader
    cpu = torch.device("cpu")
except ImportError:
    pass
dl.set_seed(42)
%matplotlib inline

---
### 🧩 Problem 9.1: Flip a batch  &nbsp; `⭐`

Write `hflip_batch(x, mask)`: `x` has shape `(B, C, H, W)`, `mask` is a boolean tensor of shape `(B,)`. Return a tensor where the images with `mask == True` are flipped left↔right (width axis) and the others are unchanged. No Python loop.


In [2]:
def hflip_batch(x, mask):
    return torch.where(mask[:, None, None, None], x.flip(3), x)

In [3]:
# ▶ self-check
x = torch.arange(2 * 1 * 2 * 3, dtype=torch.float32).reshape(2, 1, 2, 3)
out = hflip_batch(x, torch.tensor([True, False]))
dl.check("image 0 flipped", out is not None and torch.equal(out[0], x[0].flip(2)))
dl.check("image 1 untouched", out is not None and torch.equal(out[1], x[1]))

✅ image 0 flipped
✅ image 1 untouched


---
### 🧩 Problem 9.2: Shift a batch (per-image offsets)  &nbsp; `⭐⭐`

Write `shift_batch(x, dxs, dys, pad_value=0.0)`. `x` is `(B, C, H, W)`; `dxs`, `dys` are integer lists of length `B` (positive = move right / down; at most ±`m` for some small `m`).
Return the shifted batch, with vacated pixels filled by `pad_value` and nothing wrapped around.


In [4]:
def shift_batch(x, dxs, dys, pad_value=0.0):
    B, C, H, W = x.shape
    m = int(max(max(abs(d) for d in dxs), max(abs(d) for d in dys), 0))
    padded = F.pad(x, (m, m, m, m), value=pad_value)
    out = torch.empty_like(x)
    for i in range(B):
        out[i] = padded[i, :, m - dys[i]:m - dys[i] + H, m - dxs[i]:m - dxs[i] + W]
    return out

In [5]:
# ▶ self-check
x = torch.arange(1, 10, dtype=torch.float32).reshape(1, 1, 3, 3)       # [[1 2 3],[4 5 6],[7 8 9]]
r = shift_batch(torch.cat([x, x]), [1, 0], [0, -1])
dl.check("image 0 moved right by 1", r is not None and torch.equal(r[0, 0], torch.tensor([[0., 1, 2], [0, 4, 5], [0, 7, 8]])))
dl.check("image 1 moved up by 1", r is not None and torch.equal(r[1, 0], torch.tensor([[4., 5, 6], [7, 8, 9], [0, 0, 0]])))
dl.check("pad_value is honoured", torch.equal(shift_batch(x, [1], [0], pad_value=-1.0)[0, 0, :, 0], torch.tensor([-1., -1, -1])))

✅ image 0 moved right by 1
✅ image 1 moved up by 1
✅ pad_value is honoured


---
### 🧩 Problem 9.3: Top-k accuracy  &nbsp; `⭐`

Write `topk_accuracy(probs, y, k)`: the fraction of examples whose true label is among the `k` highest-probability classes. `probs` is an `(N, C)` NumPy array.


In [6]:
def topk_accuracy(probs, y, k=3):
    top = np.argsort(-probs, axis=1)[:, :k]
    return (top == y[:, None]).any(axis=1).mean()

In [7]:
# ▶ self-check
p = np.array([[.5, .3, .2], [.1, .2, .7], [.2, .5, .3]]); y = np.array([0, 1, 0])
dl.check("top-1 = 1/3", topk_accuracy(p, y, 1) is not None and np.isclose(topk_accuracy(p, y, 1), 1/3))
dl.check("top-2 = 2/3", np.isclose(topk_accuracy(p, y, 2) or 0, 2/3))
dl.check("top-3 = 1", np.isclose(topk_accuracy(p, y, 3) or 0, 1.0))

✅ top-1 = 1/3
✅ top-2 = 2/3
✅ top-3 = 1


---
### 🧩 Problem 9.4: F1 scores from a confusion matrix  &nbsp; `⭐⭐`

Given a confusion matrix `cm` (rows = true, columns = predicted), write `f1_scores(cm)` returning an array with the **F1 of every class**:
precision = TP / (column sum), recall = TP / (row sum), F1 = 2PR/(P+R). Use 0 when P+R = 0.


In [8]:
def f1_scores(cm):
    tp = np.diag(cm).astype(float)
    prec = np.divide(tp, cm.sum(0), out=np.zeros_like(tp), where=cm.sum(0) > 0)
    rec = np.divide(tp, cm.sum(1), out=np.zeros_like(tp), where=cm.sum(1) > 0)
    return np.divide(2 * prec * rec, prec + rec, out=np.zeros_like(tp), where=(prec + rec) > 0)

In [9]:
# ▶ self-check
cm = np.array([[8, 2, 0], [1, 9, 0], [0, 0, 0]])
f = f1_scores(cm)
dl.check("class 0: P=8/9, R=0.8 → F1≈0.842", f is not None and np.isclose(f[0], 2 * (8/9) * .8 / (8/9 + .8)))
dl.check("class 1: P=9/11, R=0.9", f is not None and np.isclose(f[1], 2 * (9/11) * .9 / (9/11 + .9)))
dl.check("empty class gets 0, not NaN", f is not None and f[2] == 0)

✅ class 0: P=8/9, R=0.8 → F1≈0.842
✅ class 1: P=9/11, R=0.9
✅ empty class gets 0, not NaN


---
### 🧩 Problem 9.5: Most confused pairs  &nbsp; `⭐⭐`

Write `confused_pairs(cm, names, top=3)` returning a list of `(true_name, predicted_name, count)` for the `top` largest **off-diagonal** entries, sorted by count descending.


In [10]:
def confused_pairs(cm, names, top=3):
    off = cm.copy(); np.fill_diagonal(off, 0)
    items = sorted(((off[i, j], i, j) for i in range(len(cm)) for j in range(len(cm))), reverse=True)[:top]
    return [(names[i], names[j], int(c)) for c, i, j in items]

In [11]:
# ▶ self-check
cm = np.array([[50, 7, 1], [2, 40, 9], [0, 3, 30]])
r = confused_pairs(cm, ["shirt", "coat", "bag"], top=2)
dl.check("top pair is coat→bag (9)", r is not None and r[0] == ("coat", "bag", 9))
dl.check("second pair is shirt→coat (7)", r is not None and r[1] == ("shirt", "coat", 7))

✅ top pair is coat→bag (9)
✅ second pair is shirt→coat (7)
